# Mahato/Haire 120-parameter forward model interface

This notebook mirrors the MOSS forward-model style for GSA/OAT work. It loads the 120-parameter OAT table, keeps the full Mahato/Haire model kernels in the notebook for reproducibility, and exposes `run_forward_model()` plus `mahato_forward_gfr()` for batch calls.

Document contents from papers or attached files are treated as sources only; executable behavior is defined by the code below.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
import sys
import time
import warnings

try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "mahato_sa_inputs" / "literature_range_search" / "mahato_120_parameter_ranges_oat.xlsx").exists():
    cwd_root = Path.cwd().resolve()
    if (cwd_root / "mahato_sa_inputs" / "literature_range_search" / "mahato_120_parameter_ranges_oat.xlsx").exists():
        PROJECT_ROOT = cwd_root

LOCAL_DEPS = PROJECT_ROOT / ".deps_mahato"
if LOCAL_DEPS.exists():
    sys.path.insert(0, str(LOCAL_DEPS))

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

PARAMETER_TABLE_PATH = PROJECT_ROOT / "mahato_sa_inputs" / "literature_range_search" / "mahato_120_parameter_ranges_oat.xlsx"
PARAMETER_TABLE_CSV_PATH = PARAMETER_TABLE_PATH.with_suffix(".csv")
PARAMETER_SHEET = "Parameter_Ranges_120"
PRIMARY_OUTPUT = "GFR_ml_min"
DEFAULT_SIMULATION_WEEKS = 8

print("Project:", PROJECT_ROOT)
print("Parameter table:", PARAMETER_TABLE_PATH)


In [ ]:
# ---------- solver input ----------
@dataclass(frozen=True)
class SolverConfig:
    method: str = "LSODA"
    rtol: float = 1e-12
    atol: float = 1e-12
    max_step: float = 60.0
    output_interval: float = 1440.0


SOLVER = SolverConfig()


# ---------- complete independent physiological/model parameter input ----------
BASE_PARAMETERS = {'nL_mL': 1000000.0, 'mL_nL': 1e-06, 'dl_ml': 0.01, 'L_dL': 10.0, 'L_mL': 1000.0, 'L_m3': 0.001, 'g_mg': 0.001, 'ng_mg': 1e-06, 'min_day': 1440.0, 'viscosity_length_constant': 1.5e-09, 'ECF_scale_species': 0.0004, 'BV_scale_species': 1e-06, 'water_intake_species_scale': 1.0, 'CO_scale_species': 0.0008, 'nominal_map_setpoint': 101.0, 'CO_nom': 0.004, 'ECF_nom': 0.0058, 'Na_intake_rate': 0.0004, 'nom_water_intake': 0.005, 'ref_Na_concentration': 140.0, 'plasma_protein_concentration': 4.0, 'nominal_glucose_concentration': 100.0, 'maximum_glucose_concentration': 100.0, 'T_glucose': 0.1, 'maximum_Na_intake': 0.0004, 'T_Na': 0.1, 'glucose_mg_mmol': 0.00554994394556615, 'nom_renal_threshold_glucose_reabsorption': 150.0, 'SGLT2_inhibition': 1.0, 'equilibrium_serum_creatinine': 0.8, 'P_venous': 4.0, 'R_venous': 3500.0, 'reference_BV_mean_filling_pressure': 30.18, 'nom_renal_blood_flow_L_min': 0.0018, 'baseline_nephrons': 10000.0, 'nom_Kf': 3.9, 'nom_oncotic_pressure_difference': 11.9, 'no_of_kidney': 1.0, 'nom_oncotic_pressure_peritubular': 12.0, 'interstitial_oncotic_pressure': 5.0, 'nom_preafferent_arteriole_resistance': 30000.0, 'nom_afferent_diameter': 1.3e-05, 'nom_efferent_diameter': 9.85e-06, 'Dc_pt_nom': 2.7e-05, 'Dc_lh': 1.7e-05, 'Dc_dt': 1.7e-05, 'L_pt_conv_nom': 0.0011, 'L_pt_recta_nom': 0.0011, 'L_lh_des': 0.0011, 'L_lh_asc': 0.0011, 'L_dt': 0.0023, 'tubular_compliance': 0.2, 'Pc_pt_conv_mmHg': 16.8, 'Pc_pt_recta_mmHg': 12.0, 'Pc_lh_des_mmHg': 11.0, 'Pc_lh_asc_mmHg': 10.0, 'Pc_dt_mmHg': 7.5, 'Pc_cd_mmHg': 7.8, 'nominal_pt_na_reabsorption': 0.782, 'nominal_loh_na_reabsorption': 0.4, 'nominal_dt_na_reabsorption': 0.5, 'renal_threshold_Na_reabs': 1.6e-05, 'CD_Na_reabs_threshold': 1000000.0, 'maximal_tubule_length_increase': 0.5, 'T_PT_Na_reabs_PT_length': 500000.0, 'maximal_tubule_diameter_increase': 0.5, 'T_PT_Na_reabs_PT_diameter': 500000.0, 'maximal_kf_surface_change': 0.5, 'maximum_kf_permeability_change': 1.0, 'T_kf_surface': 750.0, 'T_kf_permeability': 40000.0, 'max_preafferent_decrease': 0.25, 'T_preafferent': 4000000.0, 'nom_plasma_albumin_concentration': 3.5, 'nom_albumin_sieving_coefficient': 0.0006, 'nom_albumin_excretion': 0.075, 'PT_albumin_reabsorption_fraction': 0.995, 'max_SN_albumin_reabsorption': 2.5e-06, 'T_gp_seiving': 40000.0, 'T_gp_nephron_loss': 80000.0, 'injury_enabled': 1.0, 'afferent_dilation_enabled': 1.0, 'hypertrophy_enabled': 1.0, 'nom_rsna': 1.0, 'map_rsna_scale': 1.0, 'map_rsna_slope': 5.0, 'rsna_aff_slope': 0.7, 'rsna_aff_scale': 1.0, 'Na_controller_gain': 2.0, 'Kp_VP': 0.1, 'Ki_VP': 0.01, 'nominal_vasopressin_conc': 4.0, 'water_intake_vasopressin_scale': 1.5, 'water_intake_vasopressin_slope': -0.5, 'S_tubulo_glomerular_feedback': 0.6, 'F_md_scale_tubulo_glomerular_feedback': 0.5, 'preaff_diameter_range': 0.25, 'afferent_diameter_range': 1.2e-05, 'efferent_diameter_range': 3e-06, 'preaff_signal_nonlin_scale': 3.0, 'afferent_signal_nonlin_scale': 3.0, 'efferent_signal_nonlin_scale': 3.0, 'BV_filling_pressure_slope': 7.436, 'C_prerenal_blood_pressure': 1000.0, 'C_P_bowmans': 1000.0, 'C_P_oncotic': 1000.0, 'C_rbf': 1000.0, 'C_tgf_reset': 0.0, 'C_cardiac_output_delayed': 0.001, 'C_co_error': 1e-05, 'C_rihp': 0.1, 'C_tgf': 0.03333333333333333, 'C_na_excretion_na_amount': -0.03333333333333333, 'C_na_intake_na_amount': 0.03333333333333333, 'C_urine_flow_ecf_volume': -0.03333333333333333, 'C_water_intake_ecf_volume': 0.03333333333333333, 'C_Na_error': 0.016666666666666666, 'tissue_autoreg_scale': 1.0, 'Kp_CO': 1.5, 'Ki_CO': 30.0, 'gp_autoreg_scale': 1e-10, 'preaff_autoreg_scale': 0.1, 'myogenic_steepness': 2.0, 'RBF_autoreg_scale': 0.0, 'RBF_autoreg_steepness': 0.001, 'pressure_natriuresis_CD_scale': 0.25, 'pressure_natriuresis_CD_slope': 0.1, 'pressure_natriuresis_PT_scale': 0.0, 'pressure_natriuresis_PT_slope': 0.1, 'pressure_natriuresis_LoH_scale': 0.1, 'pressure_natriuresis_LoH_slope': 0.1, 'pressure_natriuresis_DT_scale': 0.1, 'pressure_natriuresis_DT_slope': 0.1}


UNIT_CONVERSION_CONSTANTS = {
    "nL_mL", "mL_nL", "dl_ml", "L_dL", "L_mL", "L_m3",
    "g_mg", "ng_mg", "min_day", "glucose_mg_mmol",
}

DISCRETE_MECHANISM_SWITCHES = {
    "injury_enabled", "afferent_dilation_enabled", "hypertrophy_enabled",
}

CONTINUOUS_INPUT_NAMES_120 = [
    name for name in BASE_PARAMETERS
    if name not in UNIT_CONVERSION_CONSTANTS and name not in DISCRETE_MECHANISM_SWITCHES
]

print("Code-level BASE_PARAMETERS:", len(BASE_PARAMETERS))
print("Continuous GSA/OAT inputs:", len(CONTINUOUS_INPUT_NAMES_120))
assert len(CONTINUOUS_INPUT_NAMES_120) == 120


In [ ]:
def read_parameter_table_file(path=PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET):
    path = Path(path)
    csv_path = path.with_suffix(".csv")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    try:
        return pd.read_excel(path, sheet_name=sheet_name)
    except ImportError as exc:
        if csv_path.exists() and "openpyxl" in str(exc):
            warnings.warn(f"openpyxl is unavailable; reading {csv_path} instead of {path}")
            return pd.read_csv(csv_path)
        raise
    except FileNotFoundError:
        if csv_path.exists():
            warnings.warn(f"{path} not found; reading {csv_path} instead")
            return pd.read_csv(csv_path)
        raise


def load_parameter_table(path=PARAMETER_TABLE_PATH, sheet_name=PARAMETER_SHEET):
    df = read_parameter_table_file(path, sheet_name=sheet_name)
    required = {
        "input_index", "group_id", "member_count", "members", "baseline",
        "lower_bound", "upper_bound", "unit", "category", "oat_recommendation",
    }
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Parameter table is missing required columns: {missing}")
    df = df.copy()
    df["input_index"] = df["input_index"].astype(int)
    for col in ["baseline", "lower_bound", "upper_bound"]:
        df[col] = pd.to_numeric(df[col], errors="raise")
    df = df[df["oat_recommendation"].astype(str).str.lower().eq("include")].copy()
    if len(df) != 120:
        raise ValueError(f"Expected exactly 120 continuous parameters, found {len(df)}")
    if df["group_id"].duplicated().any():
        dup = df.loc[df["group_id"].duplicated(), "group_id"].tolist()
        raise ValueError(f"Duplicate group_id values: {dup[:5]}")
    return df.sort_values("input_index").reset_index(drop=True)


def split_members(members):
    if pd.isna(members):
        return []
    return [m.strip() for m in str(members).split(";") if m.strip()]


GSA_PARAMETER_TABLE = load_parameter_table()
PARAMETER_GROUP_IDS_120 = GSA_PARAMETER_TABLE["group_id"].tolist()
PARAMETER_BASELINE_120 = dict(zip(GSA_PARAMETER_TABLE["group_id"], GSA_PARAMETER_TABLE["baseline"]))
PARAMETER_RANGES_120 = {
    row.group_id: (float(row.lower_bound), float(row.upper_bound))
    for row in GSA_PARAMETER_TABLE.itertuples(index=False)
}

print("120-parameter table loaded from:", PARAMETER_TABLE_PATH)
GSA_PARAMETER_TABLE.head()


## Embedded Mahato/Haire model kernels

The following cells are copied from the complete model notebook, with plotting/reproduction sections omitted.

In [ ]:
def calculate_dependent_parameters(parameters):
    p = dict(parameters)
    p["nom_preafferent_pressure"] = (
        p["nominal_map_setpoint"]
        - p["nom_renal_blood_flow_L_min"] * p["nom_preafferent_arteriole_resistance"]
    )
    p["nom_glomerular_pressure"] = (
        p["nom_preafferent_pressure"]
        - p["nom_renal_blood_flow_L_min"]
        * (p["L_m3"] * p["viscosity_length_constant"]
           / p["nom_afferent_diameter"]**4 / p["baseline_nephrons"])
    )
    p["nom_postglomerular_pressure"] = (
        p["nom_preafferent_pressure"]
        - p["nom_renal_blood_flow_L_min"]
        * (p["L_m3"] * p["viscosity_length_constant"]
           * (1 / p["nom_afferent_diameter"]**4 + 1 / p["nom_efferent_diameter"]**4)
           / p["baseline_nephrons"])
    )
    p["nom_GFR"] = (
        p["nom_Kf"] * (p["nom_glomerular_pressure"]
        - p["nom_oncotic_pressure_difference"] - p["Pc_pt_conv_mmHg"])
        / p["nL_mL"] * p["baseline_nephrons"]
    )
    p["nom_filtered_sodium_load"] = p["nom_GFR"] / p["L_mL"] * p["ref_Na_concentration"]
    p["nom_PT_Na_outflow"] = p["nom_filtered_sodium_load"] * (1 - p["nominal_pt_na_reabsorption"])
    p["nom_LoH_Na_outflow"] = p["nom_PT_Na_outflow"] * (1 - p["nominal_loh_na_reabsorption"])
    p["nom_DT_Na_outflow"] = p["nom_LoH_Na_outflow"] * (1 - p["nominal_dt_na_reabsorption"])
    p["nominal_cd_na_reabsorption"] = 1 - p["Na_intake_rate"] / p["nom_DT_Na_outflow"]
    p["nom_RVR"] = (p["nominal_map_setpoint"] - p["P_venous"]) / p["nom_renal_blood_flow_L_min"]
    p["nom_peritubular_resistance"] = (
        p["nom_RVR"] - p["nom_preafferent_arteriole_resistance"]
        - p["L_m3"] * p["viscosity_length_constant"]
        * (1 / p["nom_afferent_diameter"]**4 + 1 / p["nom_efferent_diameter"]**4)
        / p["baseline_nephrons"]
    )
    p["PT_Na_reab_perUnitSA_0"] = (
        (p["nom_filtered_sodium_load"] / p["baseline_nephrons"])
        * p["nominal_pt_na_reabsorption"]
        / (np.pi * p["Dc_pt_nom"] * (p["L_pt_conv_nom"] + p["L_pt_recta_nom"]))
    )
    p["nom_TPR"] = p["nominal_map_setpoint"] / p["CO_nom"]
    p["nom_systemic_arterial_resistance"] = p["nom_TPR"] - p["R_venous"]
    p["RIHP0"] = p["nom_postglomerular_pressure"]
    tubular_reabsorption = (
        p["nom_GFR"] / 1000
        - p["nom_water_intake"] * p["water_intake_species_scale"] / 1440
    )
    p["nom_peritubular_cap_Kf"] = -tubular_reabsorption / (
        p["nom_postglomerular_pressure"] - p["RIHP0"]
        - (p["nom_oncotic_pressure_peritubular"] - p["interstitial_oncotic_pressure"])
    )
    p["creatinine_synthesis_rate"] = (
        p["equilibrium_serum_creatinine"] * p["dl_ml"] * p["nom_GFR"]
    )
    return p

PARAMETERS = calculate_dependent_parameters(BASE_PARAMETERS)
print(f"Parameters: {len(PARAMETERS)}")

In [ ]:
STATE_NAMES = ['pre_renal_blood_pressure', 'tubulo_glomerular_feedback_effect', 'preafferent_pressure_autoreg_signal', 'glomerular_pressure_autoreg_signal', 'F_out_dt_delay', 'P_bowmans', 'oncotic_pressure_difference', 'renal_blood_flow_L_min_delayed', 'renal_interstitial_hydrostatic_pressure', 'tubular_length_increase', 'tubular_diameter_increase', 'kf_surface_increase', 'kf_permeability_increase', 'sieving_coeff_increase', 'fraction_nephrons_lost', 'preafferent_resistance_decrease', 'extracellular_fluid_volume', 'sodium_amount', 'cardiac_output_delayed', 'CO_error', 'Na_concentration_error', 'F0_TGF', 'serum_creatinine', 'glucose_concentration', 'Na_intake']
STATE_INDEX = {name: i for i, name in enumerate(STATE_NAMES)}

def construct_initial_conditions(p):
    ecf = p["ECF_nom"]
    blood_volume = p["BV_scale_species"] * (
        4.56 + 2.43 / (1 + np.exp(-(ecf / p["ECF_scale_species"] - 18.1) * 0.474))
    )
    values = {
        "pre_renal_blood_pressure": p["nominal_map_setpoint"],
        "tubulo_glomerular_feedback_effect": 1.0,
        "preafferent_pressure_autoreg_signal": 1.0,
        "glomerular_pressure_autoreg_signal": 1.0,
        "F_out_dt_delay": 1e-15,
        "P_bowmans": p["Pc_pt_conv_mmHg"],
        "oncotic_pressure_difference": p["nom_oncotic_pressure_difference"],
        "renal_blood_flow_L_min_delayed": p["nom_renal_blood_flow_L_min"],
        "renal_interstitial_hydrostatic_pressure": p["RIHP0"],
        "tubular_length_increase": 0.0,
        "tubular_diameter_increase": 0.0,
        "kf_surface_increase": 0.0,
        "kf_permeability_increase": 0.0,
        "sieving_coeff_increase": 0.0,
        "fraction_nephrons_lost": 0.0,
        "preafferent_resistance_decrease": 0.0,
        "extracellular_fluid_volume": ecf,
        "sodium_amount": ecf * p["ref_Na_concentration"],
        "cardiac_output_delayed": p["CO_nom"],
        "CO_error": 0.0,
        "Na_concentration_error": 0.0,
        "F0_TGF": p["nom_LoH_Na_outflow"],
        "serum_creatinine": p["equilibrium_serum_creatinine"] * blood_volume,
        "glucose_concentration": p["nominal_glucose_concentration"],
        "Na_intake": p["Na_intake_rate"],
    }
    return np.array([values[name] for name in STATE_NAMES], dtype=float)

Y0 = construct_initial_conditions(PARAMETERS)
pd.Series(Y0, index=STATE_NAMES, name="initial_value")

In [ ]:
ALGEBRAIC_OUTPUTS = ['number_of_functional_nephrons', 'number_of_functional_tubules', 'glomerular_hydrostatic_conductance_Kf', 'tissue_autoregulation_signal', 'systemic_arterial_resistance', 'total_peripheral_resistance', 'resistance_to_venous_return', 'blood_volume_L', 'mean_filling_pressure', 'venous_return', 'cardiac_output', 'mean_arterial_pressure_MAP', 'map_rsna_int', 'MAP_effect_on_rsna', 'renal_sympathetic_nerve_activity', 'preaff_arteriole_signal_multiplier', 'preaff_arteriole_adjusted_signal_multiplier', 'nom_effective_preaff_diameter', 'preaff_diameter', 'preafferent_arteriole_resistance', 'rsna_aff_intercept', 'rsna_effect_on_aff_resistance', 'afferent_arteriole_signal_multiplier', 'afferent_arteriole_adjusted_signal_multiplier', 'afferent_diameter', 'afferent_arteriole_resistance', 'efferent_arteriole_signal_multiplier', 'efferent_arteriole_adjusted_signal_multiplier', 'efferent_diameter', 'efferent_arteriole_resistance', 'RBF_autoreg_int', 'peritubular_autoreg_signal', 'autoregulated_peritubular_resistance', 'renal_vascular_resistance', 'renal_blood_flow_L_min', 'renal_blood_flow_ml_hr', 'preafferent_pressure', 'glomerular_pressure', 'postglomerular_pressure', 'gp_autoreg_int', 'preaff_autoreg_int', 'preafferent_pressure_autoreg_function', 'glomerular_pressure_autoreg_function', 'preafferent_change', 'net_filtration_pressure', 'SNGFR_nL_min', 'SNGFR_mL_min', 'GFR', 'GFR_ml_min', 'Oncotic_pressure_in', 'SNRBF_nl_min', 'plasma_protein_concentration_out', 'Oncotic_pressure_out', 'oncotic_pressure_avg', 'serum_creatinine_concentration', 'creatinine_clearance_rate', 'glucose_concentration_change', 'Na_intake_rate_change', 'filtered_glucose', 'renal_threshold_glucose_reabsorption', 'max_glucose_reabsorption', 'glucose_reabsorption', 'rate_urinary_glucose_excretion', 'SGLT2_Na_reabsorption', 'Na_concentration', 'Na_water_controller', 'normalized_vasopressin_concentration', 'vasopressin_concentration', 'water_intake_vasopressin_int', 'water_intake', 'daily_water_intake', 'SN_filtered_Na_load', 'filtered_Na_load', 'pressure_natriuresis_PT_int', 'pressure_natriuresis_PT_effect', 'pressure_natriuresis_LoH_int', 'pressure_natriuresis_LoH_effect', 'pressure_natriuresis_DT_int', 'pressure_natriuresis_DT_effect', 'pressure_natriuresis_CD_magnitude', 'pressure_natriuresis_CD_int', 'pressure_natriuresis_CD_effect', 'e_pt_sodreab', 'SN_proximal_tubule_reabsorption', 'SN_PT_Na_outflow', 'PT_Na_outflow', 'e_pt_sodreab_adj', 'e_loh_sodreab', 'SN_LoH_reabsorption', 'SN_LoH_outflow', 'SN_macula_densa_Na_flow', 'macula_densa_Na_flow', 'TGF0_tubulo_glomerular_feedback', 'tubulo_glomerular_feedback_signal', 'e_dt_sodium_reabsorption', 'SN_distal_tubule_Na_reabsorption', 'distal_tubule_Na_reabsorption', 'SN_distal_tubule_Na_outflow', 'distal_tubule_Na_outflow', 'e_cd_sodreab', 'SN_collecting_duct_Na_reabsorption', 'collecting_duct_Na_reabsorption', 'Na_excretion_via_urine', 'Na_balance', 'FENA', 'L_pt_conv', 'L_pt_recta', 'Dc_pt', 'L_pt', 'signal_glomerular_press', 'fractional_glomerular_press', 'sieving_coefficient_change', 'albumin_sieving_coefficient', 'SN_albumin_filtation_rate', 'SN_albumin_reabsorption', 'SN_albumin_excretion_rate', 'albumin_excretion_rate', 'nephron_loss_rate', 'PT_Na_reab_perUnitSA', 'normalized_PT_reabsorption_density', 'PT_Na_reabs_effect_increasing_tubular_length', 'PT_Na_reabs_effect_increasing_tubular_diameter', 'normalized_kf', 'kf_surface_change', 'kf_permeability_change', 'gamma', 'mmHg_Nperm2_conv', 'pi_model', 'A_pt', 'A_lh_des', 'A_lh_asc', 'A_dt', 'Pc_pt_conv', 'Pc_pt_recta', 'Pc_lh_des', 'Pc_lh_asc', 'Pc_dt', 'Pc_cd', 'P_interstitial', 'F_out_pt_conv', 'F_out_pt_recta', 'F_out_lh_des', 'F_out_lh_asc', 'F_out_dt', 'F_out_cd', 'urine_flow_rate', 'daily_urine_flow', 'B1', 'B2_dt', 'P_in_dt', 'P_in_dt_mmHg', 'B2_lh_asc', 'P_in_lh_asc', 'P_in_lh_asc_mmHg', 'B2_lh_des', 'P_in_lh_des', 'P_in_lh_des_mmHg', 'B2_pt_recta', 'P_in_pt_recta', 'P_in_pt_recta_mmHg', 'B2_pt_conv', 'P_in_pt_conv', 'P_in_pt_conv_mmHg', 'Oncotic_pressure_peritubular_in', 'plasma_protein_concentration_peritubular_out', 'Oncotic_pressure_peritubular_out', 'oncotic_pressure_peritubular_avg', 'tubular_reabsorption', 'RIHP']

def model_kernel(t, y, p, return_outputs=False):
    nL_mL = p["nL_mL"]
    mL_nL = p["mL_nL"]
    dl_ml = p["dl_ml"]
    L_dL = p["L_dL"]
    L_mL = p["L_mL"]
    L_m3 = p["L_m3"]
    g_mg = p["g_mg"]
    ng_mg = p["ng_mg"]
    min_day = p["min_day"]
    viscosity_length_constant = p["viscosity_length_constant"]
    ECF_scale_species = p["ECF_scale_species"]
    BV_scale_species = p["BV_scale_species"]
    water_intake_species_scale = p["water_intake_species_scale"]
    CO_scale_species = p["CO_scale_species"]
    nominal_map_setpoint = p["nominal_map_setpoint"]
    CO_nom = p["CO_nom"]
    ECF_nom = p["ECF_nom"]
    Na_intake_rate = p["Na_intake_rate"]
    nom_water_intake = p["nom_water_intake"]
    ref_Na_concentration = p["ref_Na_concentration"]
    plasma_protein_concentration = p["plasma_protein_concentration"]
    nominal_glucose_concentration = p["nominal_glucose_concentration"]
    maximum_glucose_concentration = p["maximum_glucose_concentration"]
    T_glucose = p["T_glucose"]
    maximum_Na_intake = p["maximum_Na_intake"]
    T_Na = p["T_Na"]
    glucose_mg_mmol = p["glucose_mg_mmol"]
    nom_renal_threshold_glucose_reabsorption = p["nom_renal_threshold_glucose_reabsorption"]
    SGLT2_inhibition = p["SGLT2_inhibition"]
    equilibrium_serum_creatinine = p["equilibrium_serum_creatinine"]
    P_venous = p["P_venous"]
    R_venous = p["R_venous"]
    reference_BV_mean_filling_pressure = p["reference_BV_mean_filling_pressure"]
    nom_renal_blood_flow_L_min = p["nom_renal_blood_flow_L_min"]
    baseline_nephrons = p["baseline_nephrons"]
    nom_Kf = p["nom_Kf"]
    nom_oncotic_pressure_difference = p["nom_oncotic_pressure_difference"]
    no_of_kidney = p["no_of_kidney"]
    nom_oncotic_pressure_peritubular = p["nom_oncotic_pressure_peritubular"]
    interstitial_oncotic_pressure = p["interstitial_oncotic_pressure"]
    nom_preafferent_arteriole_resistance = p["nom_preafferent_arteriole_resistance"]
    nom_afferent_diameter = p["nom_afferent_diameter"]
    nom_efferent_diameter = p["nom_efferent_diameter"]
    Dc_pt_nom = p["Dc_pt_nom"]
    Dc_lh = p["Dc_lh"]
    Dc_dt = p["Dc_dt"]
    L_pt_conv_nom = p["L_pt_conv_nom"]
    L_pt_recta_nom = p["L_pt_recta_nom"]
    L_lh_des = p["L_lh_des"]
    L_lh_asc = p["L_lh_asc"]
    L_dt = p["L_dt"]
    tubular_compliance = p["tubular_compliance"]
    Pc_pt_conv_mmHg = p["Pc_pt_conv_mmHg"]
    Pc_pt_recta_mmHg = p["Pc_pt_recta_mmHg"]
    Pc_lh_des_mmHg = p["Pc_lh_des_mmHg"]
    Pc_lh_asc_mmHg = p["Pc_lh_asc_mmHg"]
    Pc_dt_mmHg = p["Pc_dt_mmHg"]
    Pc_cd_mmHg = p["Pc_cd_mmHg"]
    nominal_pt_na_reabsorption = p["nominal_pt_na_reabsorption"]
    nominal_loh_na_reabsorption = p["nominal_loh_na_reabsorption"]
    nominal_dt_na_reabsorption = p["nominal_dt_na_reabsorption"]
    renal_threshold_Na_reabs = p["renal_threshold_Na_reabs"]
    CD_Na_reabs_threshold = p["CD_Na_reabs_threshold"]
    maximal_tubule_length_increase = p["maximal_tubule_length_increase"]
    T_PT_Na_reabs_PT_length = p["T_PT_Na_reabs_PT_length"]
    maximal_tubule_diameter_increase = p["maximal_tubule_diameter_increase"]
    T_PT_Na_reabs_PT_diameter = p["T_PT_Na_reabs_PT_diameter"]
    maximal_kf_surface_change = p["maximal_kf_surface_change"]
    maximum_kf_permeability_change = p["maximum_kf_permeability_change"]
    T_kf_surface = p["T_kf_surface"]
    T_kf_permeability = p["T_kf_permeability"]
    max_preafferent_decrease = p["max_preafferent_decrease"]
    T_preafferent = p["T_preafferent"]
    nom_plasma_albumin_concentration = p["nom_plasma_albumin_concentration"]
    nom_albumin_sieving_coefficient = p["nom_albumin_sieving_coefficient"]
    nom_albumin_excretion = p["nom_albumin_excretion"]
    PT_albumin_reabsorption_fraction = p["PT_albumin_reabsorption_fraction"]
    max_SN_albumin_reabsorption = p["max_SN_albumin_reabsorption"]
    T_gp_seiving = p["T_gp_seiving"]
    T_gp_nephron_loss = p["T_gp_nephron_loss"]
    injury_enabled = p["injury_enabled"]
    afferent_dilation_enabled = p["afferent_dilation_enabled"]
    hypertrophy_enabled = p["hypertrophy_enabled"]
    nom_rsna = p["nom_rsna"]
    map_rsna_scale = p["map_rsna_scale"]
    map_rsna_slope = p["map_rsna_slope"]
    rsna_aff_slope = p["rsna_aff_slope"]
    rsna_aff_scale = p["rsna_aff_scale"]
    Na_controller_gain = p["Na_controller_gain"]
    Kp_VP = p["Kp_VP"]
    Ki_VP = p["Ki_VP"]
    nominal_vasopressin_conc = p["nominal_vasopressin_conc"]
    water_intake_vasopressin_scale = p["water_intake_vasopressin_scale"]
    water_intake_vasopressin_slope = p["water_intake_vasopressin_slope"]
    S_tubulo_glomerular_feedback = p["S_tubulo_glomerular_feedback"]
    F_md_scale_tubulo_glomerular_feedback = p["F_md_scale_tubulo_glomerular_feedback"]
    preaff_diameter_range = p["preaff_diameter_range"]
    afferent_diameter_range = p["afferent_diameter_range"]
    efferent_diameter_range = p["efferent_diameter_range"]
    preaff_signal_nonlin_scale = p["preaff_signal_nonlin_scale"]
    afferent_signal_nonlin_scale = p["afferent_signal_nonlin_scale"]
    efferent_signal_nonlin_scale = p["efferent_signal_nonlin_scale"]
    BV_filling_pressure_slope = p["BV_filling_pressure_slope"]
    C_prerenal_blood_pressure = p["C_prerenal_blood_pressure"]
    C_P_bowmans = p["C_P_bowmans"]
    C_P_oncotic = p["C_P_oncotic"]
    C_rbf = p["C_rbf"]
    C_tgf_reset = p["C_tgf_reset"]
    C_cardiac_output_delayed = p["C_cardiac_output_delayed"]
    C_co_error = p["C_co_error"]
    C_rihp = p["C_rihp"]
    C_tgf = p["C_tgf"]
    C_na_excretion_na_amount = p["C_na_excretion_na_amount"]
    C_na_intake_na_amount = p["C_na_intake_na_amount"]
    C_urine_flow_ecf_volume = p["C_urine_flow_ecf_volume"]
    C_water_intake_ecf_volume = p["C_water_intake_ecf_volume"]
    C_Na_error = p["C_Na_error"]
    tissue_autoreg_scale = p["tissue_autoreg_scale"]
    Kp_CO = p["Kp_CO"]
    Ki_CO = p["Ki_CO"]
    gp_autoreg_scale = p["gp_autoreg_scale"]
    preaff_autoreg_scale = p["preaff_autoreg_scale"]
    myogenic_steepness = p["myogenic_steepness"]
    RBF_autoreg_scale = p["RBF_autoreg_scale"]
    RBF_autoreg_steepness = p["RBF_autoreg_steepness"]
    pressure_natriuresis_CD_scale = p["pressure_natriuresis_CD_scale"]
    pressure_natriuresis_CD_slope = p["pressure_natriuresis_CD_slope"]
    pressure_natriuresis_PT_scale = p["pressure_natriuresis_PT_scale"]
    pressure_natriuresis_PT_slope = p["pressure_natriuresis_PT_slope"]
    pressure_natriuresis_LoH_scale = p["pressure_natriuresis_LoH_scale"]
    pressure_natriuresis_LoH_slope = p["pressure_natriuresis_LoH_slope"]
    pressure_natriuresis_DT_scale = p["pressure_natriuresis_DT_scale"]
    pressure_natriuresis_DT_slope = p["pressure_natriuresis_DT_slope"]
    nom_preafferent_pressure = p["nom_preafferent_pressure"]
    nom_glomerular_pressure = p["nom_glomerular_pressure"]
    nom_postglomerular_pressure = p["nom_postglomerular_pressure"]
    nom_GFR = p["nom_GFR"]
    nom_filtered_sodium_load = p["nom_filtered_sodium_load"]
    nom_PT_Na_outflow = p["nom_PT_Na_outflow"]
    nom_LoH_Na_outflow = p["nom_LoH_Na_outflow"]
    nom_DT_Na_outflow = p["nom_DT_Na_outflow"]
    nominal_cd_na_reabsorption = p["nominal_cd_na_reabsorption"]
    nom_RVR = p["nom_RVR"]
    nom_peritubular_resistance = p["nom_peritubular_resistance"]
    PT_Na_reab_perUnitSA_0 = p["PT_Na_reab_perUnitSA_0"]
    nom_TPR = p["nom_TPR"]
    nom_systemic_arterial_resistance = p["nom_systemic_arterial_resistance"]
    RIHP0 = p["RIHP0"]
    nom_peritubular_cap_Kf = p["nom_peritubular_cap_Kf"]
    creatinine_synthesis_rate = p["creatinine_synthesis_rate"]
    pre_renal_blood_pressure = y[0]
    tubulo_glomerular_feedback_effect = y[1]
    preafferent_pressure_autoreg_signal = y[2]
    glomerular_pressure_autoreg_signal = y[3]
    F_out_dt_delay = y[4]
    P_bowmans = y[5]
    oncotic_pressure_difference = y[6]
    renal_blood_flow_L_min_delayed = y[7]
    renal_interstitial_hydrostatic_pressure = y[8]
    tubular_length_increase = y[9]
    tubular_diameter_increase = y[10]
    kf_surface_increase = y[11]
    kf_permeability_increase = y[12]
    sieving_coeff_increase = y[13]
    fraction_nephrons_lost = y[14]
    preafferent_resistance_decrease = y[15]
    extracellular_fluid_volume = y[16]
    sodium_amount = y[17]
    cardiac_output_delayed = y[18]
    CO_error = y[19]
    Na_concentration_error = y[20]
    F0_TGF = y[21]
    serum_creatinine = y[22]
    glucose_concentration = y[23]
    Na_intake = y[24]

    number_of_functional_nephrons = no_of_kidney * baseline_nephrons * (1 - 0.3 * kf_permeability_increase)
    number_of_functional_tubules = number_of_functional_nephrons * (1 - fraction_nephrons_lost)
    glomerular_hydrostatic_conductance_Kf = nom_Kf * (1 + kf_surface_increase - kf_permeability_increase)
    tissue_autoregulation_signal = max(0.1, 1 + tissue_autoreg_scale * ((Kp_CO / CO_scale_species) * (cardiac_output_delayed - CO_nom) + (Ki_CO / CO_scale_species) * CO_error))
    systemic_arterial_resistance = tissue_autoregulation_signal * nom_systemic_arterial_resistance
    total_peripheral_resistance = systemic_arterial_resistance + R_venous
    resistance_to_venous_return = (8 * R_venous + systemic_arterial_resistance) / 31
    blood_volume_L = BV_scale_species * (4.56 + 2.43 / (1 + np.exp(-(extracellular_fluid_volume / ECF_scale_species - 18.1) * 0.474)))
    mean_filling_pressure = (BV_filling_pressure_slope / BV_scale_species) * blood_volume_L - reference_BV_mean_filling_pressure
    venous_return = mean_filling_pressure / resistance_to_venous_return
    cardiac_output = venous_return
    mean_arterial_pressure_MAP = cardiac_output * total_peripheral_resistance
    map_rsna_int = 1 - map_rsna_scale / 2
    MAP_effect_on_rsna = map_rsna_int + map_rsna_scale / (1 + np.exp((mean_arterial_pressure_MAP - nominal_map_setpoint) / map_rsna_slope))
    renal_sympathetic_nerve_activity = nom_rsna * MAP_effect_on_rsna
    preaff_arteriole_signal_multiplier = preafferent_pressure_autoreg_signal
    preaff_arteriole_adjusted_signal_multiplier = 1 / (1 + np.exp(preaff_signal_nonlin_scale * (preaff_arteriole_signal_multiplier - 1))) + 0.5
    nom_effective_preaff_diameter = (nom_preafferent_arteriole_resistance * (1 - preafferent_resistance_decrease))**(-0.25)
    preaff_diameter = nom_effective_preaff_diameter + preaff_diameter_range * (preaff_arteriole_adjusted_signal_multiplier - 1)
    preafferent_arteriole_resistance = 1 / preaff_diameter**4
    rsna_aff_intercept = 1 - rsna_aff_scale / 2
    rsna_effect_on_aff_resistance = max(0.25, rsna_aff_intercept + rsna_aff_scale / (1 + np.exp((1 - renal_sympathetic_nerve_activity) / rsna_aff_slope)))
    afferent_arteriole_signal_multiplier = rsna_effect_on_aff_resistance * tubulo_glomerular_feedback_effect * glomerular_pressure_autoreg_signal
    afferent_arteriole_adjusted_signal_multiplier = 1 / (1 + np.exp(afferent_signal_nonlin_scale * (afferent_arteriole_signal_multiplier - 1))) + 0.5
    afferent_diameter = nom_afferent_diameter * (1 - preafferent_resistance_decrease)**(-0.25) + afferent_diameter_range * (afferent_arteriole_adjusted_signal_multiplier - 1)
    afferent_arteriole_resistance = L_m3 * viscosity_length_constant / afferent_diameter**4
    efferent_arteriole_signal_multiplier = 1
    efferent_arteriole_adjusted_signal_multiplier = 1 / (1 + np.exp(efferent_signal_nonlin_scale * (efferent_arteriole_signal_multiplier - 1))) + 0.5
    efferent_diameter = nom_efferent_diameter + efferent_diameter_range * (efferent_arteriole_adjusted_signal_multiplier - 1)
    efferent_arteriole_resistance = L_m3 * viscosity_length_constant / efferent_diameter**4
    RBF_autoreg_int = 1 - RBF_autoreg_scale / 2
    peritubular_autoreg_signal = RBF_autoreg_int + RBF_autoreg_scale / (1 + np.exp((nom_renal_blood_flow_L_min - renal_blood_flow_L_min_delayed) / RBF_autoreg_steepness))
    autoregulated_peritubular_resistance = peritubular_autoreg_signal * nom_peritubular_resistance
    renal_vascular_resistance = preafferent_arteriole_resistance + (afferent_arteriole_resistance + efferent_arteriole_resistance) / (number_of_functional_nephrons / no_of_kidney) + autoregulated_peritubular_resistance
    renal_blood_flow_L_min = (pre_renal_blood_pressure - P_venous) / renal_vascular_resistance
    renal_blood_flow_ml_hr = renal_blood_flow_L_min * 1000 * 60
    preafferent_pressure = mean_arterial_pressure_MAP - renal_blood_flow_L_min * preafferent_arteriole_resistance
    glomerular_pressure = pre_renal_blood_pressure - renal_blood_flow_L_min * (preafferent_arteriole_resistance + afferent_arteriole_resistance / (number_of_functional_nephrons / no_of_kidney))
    postglomerular_pressure = pre_renal_blood_pressure - renal_blood_flow_L_min * (preafferent_arteriole_resistance + (afferent_arteriole_resistance + efferent_arteriole_resistance) / (number_of_functional_nephrons / no_of_kidney))
    gp_autoreg_int = 1 - gp_autoreg_scale / 2
    preaff_autoreg_int = 1 - preaff_autoreg_scale / 2
    preafferent_pressure_autoreg_function = preaff_autoreg_int + preaff_autoreg_scale / (1 + np.exp((nom_preafferent_pressure - preafferent_pressure) / myogenic_steepness))
    glomerular_pressure_autoreg_function = gp_autoreg_int + gp_autoreg_scale / (1 + np.exp((nom_glomerular_pressure - glomerular_pressure) / myogenic_steepness))
    preafferent_change = afferent_dilation_enabled * (max_preafferent_decrease - preafferent_resistance_decrease) * max(glucose_concentration - 120, 0) / T_preafferent
    net_filtration_pressure = glomerular_pressure - oncotic_pressure_difference - P_bowmans
    SNGFR_nL_min = glomerular_hydrostatic_conductance_Kf * net_filtration_pressure
    SNGFR_mL_min = SNGFR_nL_min * mL_nL
    GFR = SNGFR_nL_min / 1000 / 1000000 * number_of_functional_tubules
    GFR_ml_min = GFR * 1000
    Oncotic_pressure_in = 1.629 * plasma_protein_concentration + 0.2935 * plasma_protein_concentration**2
    SNRBF_nl_min = 1e9 * renal_blood_flow_L_min / number_of_functional_nephrons
    plasma_protein_concentration_out = SNRBF_nl_min * plasma_protein_concentration / (SNRBF_nl_min - SNGFR_nL_min)
    Oncotic_pressure_out = 1.629 * plasma_protein_concentration_out + 0.2935 * plasma_protein_concentration_out**2
    oncotic_pressure_avg = (Oncotic_pressure_in + Oncotic_pressure_out) / 2
    serum_creatinine_concentration = serum_creatinine / blood_volume_L
    creatinine_clearance_rate = GFR_ml_min * dl_ml * serum_creatinine_concentration
    glucose_concentration_change = max(maximum_glucose_concentration - glucose_concentration, 0) / T_glucose
    Na_intake_rate_change = max(maximum_Na_intake - Na_intake, 0) / T_Na
    filtered_glucose = GFR * glucose_concentration * 10
    renal_threshold_glucose_reabsorption = nom_renal_threshold_glucose_reabsorption * SGLT2_inhibition
    max_glucose_reabsorption = renal_threshold_glucose_reabsorption * GFR * 10
    glucose_reabsorption = min(max_glucose_reabsorption, filtered_glucose)
    rate_urinary_glucose_excretion = max(0, filtered_glucose - max_glucose_reabsorption) * 1440 / 1000
    SGLT2_Na_reabsorption = glucose_reabsorption * glucose_mg_mmol
    Na_concentration = sodium_amount / extracellular_fluid_volume
    Na_water_controller = Na_controller_gain * (Kp_VP * (Na_concentration - ref_Na_concentration) + Ki_VP * Na_concentration_error)
    normalized_vasopressin_concentration = 1 + Na_water_controller
    vasopressin_concentration = nominal_vasopressin_conc * normalized_vasopressin_concentration
    water_intake_vasopressin_int = 1 - water_intake_vasopressin_scale / 2
    water_intake = water_intake_species_scale * (nom_water_intake / 1440) * (water_intake_vasopressin_int + water_intake_vasopressin_scale / (1 + np.exp((normalized_vasopressin_concentration - 1) / water_intake_vasopressin_slope)))
    daily_water_intake = water_intake * 1440
    SN_filtered_Na_load = SNGFR_nL_min / 1e9 * Na_concentration
    filtered_Na_load = SN_filtered_Na_load * number_of_functional_tubules
    pressure_natriuresis_PT_int = 1 - pressure_natriuresis_PT_scale / 2
    pressure_natriuresis_PT_effect = max(0.001, pressure_natriuresis_PT_int + pressure_natriuresis_PT_scale / (1 + np.exp((postglomerular_pressure - RIHP0) / pressure_natriuresis_PT_slope)))
    pressure_natriuresis_LoH_int = 1 - pressure_natriuresis_LoH_scale / 2
    pressure_natriuresis_LoH_effect = max(0.001, pressure_natriuresis_LoH_int + pressure_natriuresis_LoH_scale / (1 + np.exp((postglomerular_pressure - RIHP0) / pressure_natriuresis_LoH_slope)))
    pressure_natriuresis_DT_int = 1 - pressure_natriuresis_DT_scale / 2
    pressure_natriuresis_DT_effect = max(0.001, pressure_natriuresis_DT_int + pressure_natriuresis_DT_scale / (1 + np.exp((postglomerular_pressure - RIHP0) / pressure_natriuresis_DT_slope)))
    pressure_natriuresis_CD_magnitude = max(0, pressure_natriuresis_CD_scale)
    pressure_natriuresis_CD_int = 1 - pressure_natriuresis_CD_magnitude / 2
    pressure_natriuresis_CD_effect = max(0.001, pressure_natriuresis_CD_int + pressure_natriuresis_CD_magnitude / (1 + np.exp((postglomerular_pressure - RIHP0) / pressure_natriuresis_CD_slope)))
    e_pt_sodreab = nominal_pt_na_reabsorption * pressure_natriuresis_PT_effect
    SN_proximal_tubule_reabsorption = min(SN_filtered_Na_load, renal_threshold_Na_reabs) * e_pt_sodreab + (SGLT2_Na_reabsorption - 0.00235) / number_of_functional_tubules
    SN_PT_Na_outflow = max(0, SN_filtered_Na_load - SN_proximal_tubule_reabsorption)
    PT_Na_outflow = SN_PT_Na_outflow * number_of_functional_tubules
    e_pt_sodreab_adj = SN_proximal_tubule_reabsorption / SN_filtered_Na_load
    e_loh_sodreab = nominal_loh_na_reabsorption * (1 - 0.65) * pressure_natriuresis_LoH_effect
    SN_LoH_reabsorption = SN_PT_Na_outflow * nominal_loh_na_reabsorption * pressure_natriuresis_LoH_effect
    SN_LoH_outflow = SN_PT_Na_outflow * (1 - nominal_loh_na_reabsorption * pressure_natriuresis_LoH_effect)
    SN_macula_densa_Na_flow = SN_LoH_outflow
    macula_densa_Na_flow = SN_macula_densa_Na_flow * number_of_functional_tubules
    TGF0_tubulo_glomerular_feedback = 1 - S_tubulo_glomerular_feedback / 2
    tubulo_glomerular_feedback_signal = TGF0_tubulo_glomerular_feedback + S_tubulo_glomerular_feedback / (1 + np.exp((F0_TGF - SN_macula_densa_Na_flow * baseline_nephrons) * (2e6 / baseline_nephrons) / F_md_scale_tubulo_glomerular_feedback))
    e_dt_sodium_reabsorption = nominal_dt_na_reabsorption * pressure_natriuresis_DT_effect
    SN_distal_tubule_Na_reabsorption = SN_macula_densa_Na_flow * min(1, e_dt_sodium_reabsorption)
    distal_tubule_Na_reabsorption = SN_distal_tubule_Na_reabsorption * number_of_functional_tubules
    SN_distal_tubule_Na_outflow = SN_macula_densa_Na_flow - SN_distal_tubule_Na_reabsorption
    distal_tubule_Na_outflow = SN_distal_tubule_Na_outflow * number_of_functional_tubules
    e_cd_sodreab = min(1, nominal_cd_na_reabsorption * pressure_natriuresis_CD_effect)
    SN_collecting_duct_Na_reabsorption = min(CD_Na_reabs_threshold, SN_distal_tubule_Na_outflow * min(1, e_cd_sodreab))
    collecting_duct_Na_reabsorption = SN_collecting_duct_Na_reabsorption * number_of_functional_tubules
    Na_excretion_via_urine = (SN_distal_tubule_Na_outflow - SN_collecting_duct_Na_reabsorption) * number_of_functional_tubules
    Na_balance = Na_intake - Na_excretion_via_urine
    FENA = Na_excretion_via_urine / filtered_Na_load
    L_pt_conv = L_pt_conv_nom * (1 + tubular_length_increase)
    L_pt_recta = L_pt_recta_nom * (1 + tubular_length_increase)
    Dc_pt = Dc_pt_nom * (1 + tubular_diameter_increase)
    L_pt = L_pt_conv + L_pt_recta
    signal_glomerular_press = glomerular_pressure - nom_glomerular_pressure
    fractional_glomerular_press = max(0, (glomerular_pressure - 2) / nom_glomerular_pressure - 1)
    sieving_coefficient_change = injury_enabled * fractional_glomerular_press / T_gp_seiving
    albumin_sieving_coefficient = nom_albumin_sieving_coefficient + sieving_coeff_increase
    SN_albumin_filtation_rate = nom_plasma_albumin_concentration * SNGFR_nL_min * albumin_sieving_coefficient * mL_nL * dl_ml * 1e6
    SN_albumin_reabsorption = min(max_SN_albumin_reabsorption * 1000, SN_albumin_filtation_rate * PT_albumin_reabsorption_fraction)
    SN_albumin_excretion_rate = (SN_albumin_filtation_rate - SN_albumin_reabsorption) * number_of_functional_tubules * 1440 / 1000
    albumin_excretion_rate = nom_albumin_excretion + SN_albumin_excretion_rate
    nephron_loss_rate = injury_enabled * (1 - fraction_nephrons_lost) * max(0, fractional_glomerular_press) / T_gp_nephron_loss
    PT_Na_reab_perUnitSA = SN_filtered_Na_load * e_pt_sodreab_adj / (3.14 * Dc_pt * (L_pt_conv + L_pt_recta))
    normalized_PT_reabsorption_density = PT_Na_reab_perUnitSA / PT_Na_reab_perUnitSA_0
    PT_Na_reabs_effect_increasing_tubular_length = (maximal_tubule_length_increase - tubular_length_increase) * max(normalized_PT_reabsorption_density - 1, 0) / T_PT_Na_reabs_PT_length
    PT_Na_reabs_effect_increasing_tubular_diameter = (maximal_tubule_diameter_increase - tubular_diameter_increase) * max(normalized_PT_reabsorption_density - 1, 0) / T_PT_Na_reabs_PT_diameter
    normalized_kf = (glomerular_pressure - 1.05) / nom_glomerular_pressure
    kf_surface_change = hypertrophy_enabled * (maximal_kf_surface_change - kf_surface_increase) * max(normalized_kf - 1, 0) / T_kf_surface
    kf_permeability_change = injury_enabled * (maximum_kf_permeability_change - kf_permeability_increase) * max(normalized_kf - 1, 0) / T_kf_permeability
    gamma = 1.16667e-5
    mmHg_Nperm2_conv = 133.32
    pi_model = 3.14
    A_pt = -np.log(1 - e_pt_sodreab) / (L_pt_conv + L_pt_recta)
    A_lh_des = -np.log(1 - nominal_loh_na_reabsorption) / L_lh_des
    A_lh_asc = 1
    A_dt = -np.log(1 - e_dt_sodium_reabsorption) / L_dt
    Pc_pt_conv = Pc_pt_conv_mmHg * mmHg_Nperm2_conv
    Pc_pt_recta = Pc_pt_recta_mmHg * mmHg_Nperm2_conv
    Pc_lh_des = Pc_lh_des_mmHg * mmHg_Nperm2_conv
    Pc_lh_asc = Pc_lh_asc_mmHg * mmHg_Nperm2_conv
    Pc_dt = Pc_dt_mmHg * mmHg_Nperm2_conv
    Pc_cd = Pc_cd_mmHg * mmHg_Nperm2_conv
    P_interstitial = nom_postglomerular_pressure * mmHg_Nperm2_conv
    F_out_pt_conv = SNGFR_nL_min * 1e-12 * np.exp(-A_pt * L_pt_conv)
    F_out_pt_recta = F_out_pt_conv * np.exp(-A_pt * L_pt_recta)
    F_out_lh_des = F_out_pt_recta * np.exp(-A_lh_des * L_lh_des)
    F_out_lh_asc = F_out_lh_des
    F_out_dt = F_out_lh_asc * np.exp(-A_dt * L_dt)
    F_out_cd = max(F_out_dt * (1 - e_cd_sodreab * normalized_vasopressin_concentration), 0)
    urine_flow_rate = F_out_cd * 1e12 * number_of_functional_tubules * 1e-9
    daily_urine_flow = urine_flow_rate * 1440
    B1 = (4 * tubular_compliance + 1) * 128 * gamma / pi_model
    B2_dt = Pc_dt**(4 * tubular_compliance) * F_out_lh_asc / (Dc_dt**4 * A_dt)
    P_in_dt = (Pc_cd**(4 * tubular_compliance + 1) + B1 * B2_dt * (1 - np.exp(-A_dt * L_dt)))** (1 / (4 * tubular_compliance + 1))
    P_in_dt_mmHg = (P_in_dt + P_interstitial) / mmHg_Nperm2_conv
    B2_lh_asc = Pc_lh_asc**(4 * tubular_compliance) * F_out_lh_des / (Dc_lh**4 * A_lh_asc)
    P_in_lh_asc = (P_in_dt**(4 * tubular_compliance + 1) + B1 * B2_lh_asc * (1 - np.exp(-A_lh_asc * L_lh_asc)))** (1 / (4 * tubular_compliance + 1))
    P_in_lh_asc_mmHg = (P_in_lh_asc + P_interstitial) / mmHg_Nperm2_conv
    B2_lh_des = Pc_lh_des**(4 * tubular_compliance) * F_out_pt_recta / (Dc_lh**4 * A_lh_des)
    P_in_lh_des = (P_in_lh_asc**(4 * tubular_compliance + 1) + B1 * B2_lh_des * (1 - np.exp(-A_lh_des * L_lh_des)))** (1 / (4 * tubular_compliance + 1))
    P_in_lh_des_mmHg = (P_in_lh_des + P_interstitial) / mmHg_Nperm2_conv
    B2_pt_recta = Pc_pt_recta**(4 * tubular_compliance) * F_out_pt_conv / (Dc_pt**4 * A_pt)
    P_in_pt_recta = (P_in_lh_des**(4 * tubular_compliance + 1) + B1 * B2_pt_recta * (1 - np.exp(-A_pt * L_pt_recta)))** (1 / (4 * tubular_compliance + 1))
    P_in_pt_recta_mmHg = (P_in_pt_recta + P_interstitial) / mmHg_Nperm2_conv
    B2_pt_conv = Pc_pt_conv**(4 * tubular_compliance) * (SNGFR_nL_min * 1e-12) / (Dc_pt**4 * A_pt)
    P_in_pt_conv = (P_in_pt_recta**(4 * tubular_compliance + 1) + B1 * B2_pt_conv * (1 - np.exp(-A_pt * L_pt_conv)))** (1 / (4 * tubular_compliance + 1))
    P_in_pt_conv_mmHg = (P_in_pt_conv + P_interstitial) / mmHg_Nperm2_conv
    Oncotic_pressure_peritubular_in = Oncotic_pressure_out
    plasma_protein_concentration_peritubular_out = SNRBF_nl_min * plasma_protein_concentration / (SNRBF_nl_min - urine_flow_rate * 1e9 / number_of_functional_nephrons)
    Oncotic_pressure_peritubular_out = 1.629 * plasma_protein_concentration_peritubular_out + 0.2935 * plasma_protein_concentration_peritubular_out**2
    oncotic_pressure_peritubular_avg = (Oncotic_pressure_peritubular_in + Oncotic_pressure_peritubular_out) / 2
    tubular_reabsorption = GFR_ml_min / 1000 - urine_flow_rate
    RIHP = postglomerular_pressure - (oncotic_pressure_peritubular_avg - interstitial_oncotic_pressure) + tubular_reabsorption / nom_peritubular_cap_Kf

    dy = np.array([
        C_prerenal_blood_pressure * (mean_arterial_pressure_MAP - pre_renal_blood_pressure),
        C_tgf * (tubulo_glomerular_feedback_signal - tubulo_glomerular_feedback_effect),
        500 * (preafferent_pressure_autoreg_function - preafferent_pressure_autoreg_signal),
        500 * (glomerular_pressure_autoreg_function - glomerular_pressure_autoreg_signal),
        100 * (F_out_dt - F_out_dt_delay),
        C_P_bowmans * (P_in_pt_conv_mmHg - P_bowmans),
        C_P_oncotic * (oncotic_pressure_avg - oncotic_pressure_difference),
        C_rbf * (renal_blood_flow_L_min - renal_blood_flow_L_min_delayed),
        C_rihp * (RIHP - renal_interstitial_hydrostatic_pressure),
        PT_Na_reabs_effect_increasing_tubular_length,
        PT_Na_reabs_effect_increasing_tubular_diameter,
        kf_surface_change,
        kf_permeability_change,
        sieving_coefficient_change,
        nephron_loss_rate,
        preafferent_change,
        C_water_intake_ecf_volume * water_intake + C_urine_flow_ecf_volume * urine_flow_rate,
        C_na_excretion_na_amount * Na_excretion_via_urine + C_na_intake_na_amount * Na_intake,
        C_cardiac_output_delayed * (cardiac_output - cardiac_output_delayed),
        C_co_error * (cardiac_output - CO_nom),
        C_Na_error * (Na_concentration - ref_Na_concentration),
        C_tgf_reset * (SN_macula_densa_Na_flow * baseline_nephrons - F0_TGF),
        creatinine_synthesis_rate - creatinine_clearance_rate,
        glucose_concentration_change,
        Na_intake_rate_change
    ], dtype=float)
    if return_outputs:
        local_values = locals()
        outputs = {name: float(local_values[name]) for name in ALGEBRAIC_OUTPUTS}
        outputs.update({name: float(y[i]) for i, name in enumerate(STATE_NAMES)})
        return dy, outputs
    return dy

dy0, outputs0 = model_kernel(0.0, Y0, PARAMETERS, return_outputs=True)
assert dy0.shape == (25,)
assert np.isfinite(dy0).all()
print("Mechanism outputs:", len(outputs0), "| dynamic states:", len(dy0))

In [ ]:
MINUTES_PER_WEEK = 7 * 1440

def with_overrides(base, overrides=None):
    p = dict(base)
    if overrides:
        unknown = set(overrides) - set(p)
        if unknown:
            raise KeyError(f"Unknown parameter overrides: {sorted(unknown)}")
        p.update(overrides)
    return p

def evaluate_trajectory(times, states, p, scenario, event_phase=None):
    rows = []
    for i, time in enumerate(times):
        _, outputs = model_kernel(time, states[:, i], p, return_outputs=True)
        outputs["time"] = float(time)
        outputs["time_minutes"] = float(time)
        outputs["time_weeks"] = float(time / MINUTES_PER_WEEK)
        outputs["scenario"] = scenario
        outputs["event_phase"] = "" if event_phase is None else event_phase[i]
        rows.append(outputs)
    return pd.DataFrame(rows)

def run_scenario(name, base_parameters, initial_state, output_times,
                 overrides=None, solver=SOLVER):
    p = with_overrides(base_parameters, overrides)
    times = np.unique(np.asarray(output_times, dtype=float))
    solution = solve_ivp(
        lambda t, y: model_kernel(t, y, p),
        (float(times[0]), float(times[-1])),
        np.asarray(initial_state, dtype=float),
        method=solver.method, t_eval=times, rtol=solver.rtol,
        atol=solver.atol, max_step=solver.max_step,
    )
    if not solution.success:
        raise RuntimeError(f"{name}: {solution.message}")
    if not np.isfinite(solution.y).all():
        raise FloatingPointError(f"{name}: non-finite state")
    return evaluate_trajectory(solution.t, solution.y, p, name)

def final_state(frame):
    return frame.iloc[-1][STATE_NAMES].to_numpy(dtype=float)

def run_two_stage_event(name, base_parameters, initial_state, output_times,
                        event_time, before_overrides, after_overrides,
                        solver=SOLVER):
    output_times = np.asarray(output_times, dtype=float)
    before_times = np.unique(np.r_[output_times[output_times <= event_time], event_time])
    before = run_scenario(name, base_parameters, initial_state, before_times,
                          before_overrides, solver)
    after_absolute = np.unique(np.r_[event_time, output_times[output_times >= event_time]])
    after_relative = after_absolute - event_time
    after = run_scenario(name, base_parameters, final_state(before),
                         after_relative, after_overrides, solver)
    after["time"] += event_time
    after["time_minutes"] = after["time"]
    after["time_weeks"] = after["time"] / MINUTES_PER_WEEK
    before.loc[before["time_minutes"] == event_time, "event_phase"] = "pre_UNX"
    after.loc[after["time_minutes"] == event_time, "event_phase"] = "post_UNX"
    return pd.concat([before, after], ignore_index=True)

In [ ]:
BASELINE_OVERRIDES = {
    "afferent_dilation_enabled": 0, "hypertrophy_enabled": 0, "injury_enabled": 0
}
DIABETES_OVERRIDES = {
    "maximum_glucose_concentration": 500, "T_glucose": 50000,
    "maximum_Na_intake": 0.0005, "T_Na": 50000,
    "nom_renal_threshold_glucose_reabsorption": 400,
    "nom_water_intake": 0.025,
}
INJURY_OVERRIDES = {
    "T_kf_permeability": 70000,
    "T_gp_nephron_loss": 80000,
    "T_gp_seiving": 500000,
}

FIGURE3_SCENARIOS = {
    "control": BASELINE_OVERRIDES,
    "diabetes_no_adaptation": {
        **DIABETES_OVERRIDES, "afferent_dilation_enabled": 0,
        "hypertrophy_enabled": 0, "injury_enabled": 0,
    },
    "diabetes_afferent_dilation": {
        **DIABETES_OVERRIDES, "afferent_dilation_enabled": 1,
        "hypertrophy_enabled": 0, "injury_enabled": 0,
    },
    "diabetes_afferent_dilation_hypertrophy": {
        **DIABETES_OVERRIDES, "afferent_dilation_enabled": 1,
        "hypertrophy_enabled": 1, "injury_enabled": 0,
    },
}

def add_reproduction_outputs(frame, p):
    result = frame.copy()
    result["Na_excretion_mEq_day"] = result["Na_excretion_via_urine"] * 1440
    result["Na_balance_mEq_day"] = result["Na_balance"] * 1440
    result["RBF_ml_min"] = result["renal_blood_flow_L_min"] * 1000
    result["distal_fractional_na_reabs_normalized"] = (
        result["e_dt_sodium_reabsorption"] / p["nominal_dt_na_reabsorption"]
    )
    result["preglomerular_resistance_k"] = (
        result["preafferent_arteriole_resistance"]
        + result["afferent_arteriole_resistance"]
        / result["number_of_functional_nephrons"]
    ) / 1000
    result["Kf_surface_component"] = p["nom_Kf"] * result["kf_surface_increase"]
    result["Kf_permeability_component"] = -p["nom_Kf"] * result["kf_permeability_increase"]
    return result

## Forward-model helpers

Grouped-parameter interface for OAT/GSA. The default scenario is the control baseline with the three mechanism switches off.

In [ ]:
def validate_parameter_table_against_model(parameter_table=None, atol=1e-12, rtol=1e-9):
    if parameter_table is None:
        parameter_table = GSA_PARAMETER_TABLE
    rows = []
    control_base = {**BASE_PARAMETERS, **BASELINE_OVERRIDES}
    for _, row in parameter_table.iterrows():
        members = split_members(row["members"])
        values = []
        errors = []
        for member in members:
            if member not in control_base:
                errors.append(f"{member}: not found in BASE_PARAMETERS")
            else:
                values.append(float(control_base[member]))
        expected = float(row["baseline"])
        ok = (not errors) and len(values) == int(row["member_count"]) and all(
            np.isclose(v, expected, atol=atol, rtol=rtol) for v in values
        )
        rows.append({
            "input_index": int(row["input_index"]),
            "group_id": row["group_id"],
            "members": row["members"],
            "member_count_declared": int(row["member_count"]),
            "member_count_found": len(values),
            "baseline_expected": expected,
            "min_member_baseline": min(values) if values else np.nan,
            "max_member_baseline": max(values) if values else np.nan,
            "recognized_by_model": not errors,
            "baseline_matches_model": ok,
            "validation_errors": " | ".join(errors),
        })
    return pd.DataFrame(rows)


def _row_for_parameter_key(key, parameter_table):
    table_by_group = parameter_table.set_index("group_id", drop=False)
    table_by_member = parameter_table.set_index("members", drop=False)
    table_by_index = parameter_table.set_index("input_index", drop=False)
    if key in table_by_group.index:
        return table_by_group.loc[key]
    if key in table_by_member.index:
        return table_by_member.loc[key]
    if isinstance(key, int) and key in table_by_index.index:
        return table_by_index.loc[key]
    if isinstance(key, str):
        try:
            as_int = int(key)
        except ValueError:
            as_int = None
        if as_int is not None and as_int in table_by_index.index:
            return table_by_index.loc[as_int]
    raise KeyError(f"Unknown Mahato grouped parameter: {key}")


def apply_grouped_parameter(parameters, parameter_row, value):
    for member in split_members(parameter_row["members"]):
        if member not in parameters:
            raise KeyError(f"Unknown BASE_PARAMETERS member: {member}")
        parameters[member] = float(value)
    return parameters


def build_parameters_from_parameter_values(
    parameter_values=None,
    parameter_table=None,
    scenario_overrides=None,
):
    """Build a dependent parameter dictionary from grouped GSA/OAT inputs.

    parameter_values may be keyed by group_id, member name, or integer input_index.
    The default scenario is the control baseline: the three mechanism switches are off.
    """
    if parameter_table is None:
        parameter_table = GSA_PARAMETER_TABLE
    parameters = dict(BASE_PARAMETERS)
    parameters.update(BASELINE_OVERRIDES)
    if scenario_overrides:
        unknown = set(scenario_overrides) - set(parameters)
        if unknown:
            raise KeyError(f"Unknown scenario overrides: {sorted(unknown)}")
        parameters.update(scenario_overrides)
    if parameter_values:
        for key, value in parameter_values.items():
            row = _row_for_parameter_key(key, parameter_table)
            apply_grouped_parameter(parameters, row, value)
    return calculate_dependent_parameters(parameters)


def default_output_times(simulation_weeks=DEFAULT_SIMULATION_WEEKS, solver=SOLVER):
    return np.arange(0, simulation_weeks * MINUTES_PER_WEEK + 1, solver.output_interval)


def run_model_instance(
    parameters,
    output_times=None,
    solver=SOLVER,
    name="mahato_forward",
    return_trajectory=False,
):
    if output_times is None:
        output_times = default_output_times(solver=solver)
    initial_state = construct_initial_conditions(parameters)
    started = time.time()
    trajectory = run_scenario(
        name=name,
        base_parameters=parameters,
        initial_state=initial_state,
        output_times=output_times,
        overrides=None,
        solver=solver,
    )
    trajectory = add_reproduction_outputs(trajectory, parameters)
    runtime_seconds = time.time() - started
    final = trajectory.iloc[-1].copy()
    final["runtime_seconds"] = runtime_seconds
    final["status"] = "success"
    final["message"] = ""
    if return_trajectory:
        return final, trajectory
    return final


def run_forward_model(
    parameter_values=None,
    parameter_table=None,
    scenario_overrides=None,
    output_times=None,
    solver=SOLVER,
    return_details=False,
):
    parameters = build_parameters_from_parameter_values(
        parameter_values=parameter_values,
        parameter_table=parameter_table,
        scenario_overrides=scenario_overrides,
    )
    result = run_model_instance(
        parameters=parameters,
        output_times=output_times,
        solver=solver,
        return_trajectory=return_details,
    )
    if return_details:
        final, trajectory = result
        return final, trajectory, parameters
    return result


def mahato_forward_gfr(parameter_values=None, parameter_table=None, scenario_overrides=None):
    """Run the Mahato/Haire forward model and return only GFR_ml_min.

    Examples:
        mahato_forward_gfr({"nom_renal_blood_flow_L_min": 0.002})
        mahato_forward_gfr({"nom_renal_blood_flow_L_min|baseline=0.0018": 0.002})
        mahato_forward_gfr({31: 0.002})
    """
    result = run_forward_model(
        parameter_values=parameter_values,
        parameter_table=parameter_table,
        scenario_overrides=scenario_overrides,
        return_details=False,
    )
    return float(result[PRIMARY_OUTPUT])


def run_oat_row(row, parameter_table=None, solver=SOLVER):
    """Run one row from an OAT matrix with columns parameter_name and test_value."""
    if parameter_table is None:
        parameter_table = GSA_PARAMETER_TABLE
    key = row["parameter_name"] if "parameter_name" in row else row["members"]
    value = float(row["test_value"])
    started = time.time()
    try:
        final = run_forward_model(
            parameter_values={key: value},
            parameter_table=parameter_table,
            solver=solver,
            return_details=False,
        )
        return {
            "parameter_name": key,
            "test_value": value,
            "status": "success",
            "message": "",
            "GFR_ml_min": float(final["GFR_ml_min"]),
            "GFR": float(final["GFR"]),
            "MAP": float(final["mean_arterial_pressure_MAP"]),
            "RBF_ml_min": float(final["RBF_ml_min"]),
            "SNGFR_nL_min": float(final["SNGFR_nL_min"]),
            "Na_excretion_mEq_day": float(final["Na_excretion_mEq_day"]),
            "runtime_seconds": time.time() - started,
        }
    except Exception as exc:
        return {
            "parameter_name": key,
            "test_value": value,
            "status": "failed",
            "message": f"{type(exc).__name__}: {exc}",
            "GFR_ml_min": np.nan,
            "GFR": np.nan,
            "MAP": np.nan,
            "RBF_ml_min": np.nan,
            "SNGFR_nL_min": np.nan,
            "Na_excretion_mEq_day": np.nan,
            "runtime_seconds": time.time() - started,
        }


PARAMETER_VALIDATION = validate_parameter_table_against_model(GSA_PARAMETER_TABLE)
print("Grouped parameters loaded:", len(GSA_PARAMETER_TABLE))
print("All 120 recognized and baseline-matched:", bool(PARAMETER_VALIDATION["baseline_matches_model"].all()))


In [ ]:
baseline_check = run_forward_model(return_details=False)
baseline_check[[
    "GFR_ml_min",
    "GFR",
    "SNGFR_nL_min",
    "mean_arterial_pressure_MAP",
    "RBF_ml_min",
    "Na_excretion_mEq_day",
    "runtime_seconds",
    "status",
]]
